In [20]:
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Bidirectional, Dense, Dropout,
    LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model

Arabic normalization

In [21]:
def normalize_arabic(text):
    text = str(text)

    # Remove tashkeel
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)

    # Normalize Alef
    text = re.sub(r'[إأآا]', 'ا', text)

    # Normalize Ya
    text = re.sub(r'ى', 'ي', text)

    # Normalize Ta Marbuta
    text = re.sub(r'ة', 'ه', text)

    # Remove Arabic commas, question marks, and other non-alphanumeric chars
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove punctuation except Arabic/English letters and numbers
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # remove Arabic + English punctuation
    text = re.sub(r'[،؛؟,.!?:"“”()\[\]{}\-ـ]', ' ', text)

    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [22]:
def load_qa_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for article in raw.get("data", []):
        for paragraph in article.get("paragraphs", []):
            context = paragraph.get("context", "")
            for qa in paragraph.get("qas", []):
                question = qa.get("question", "")
                answers = qa.get("answers", [])
                answer_text = answers[0].get("text", "") if answers else ""

                rows.append({
                    "id": qa.get("id", ""),
                    "question": question,
                    "context": context,
                    "answer": answer_text
                })

    return rows

DATASET_DIR = "data_MS2"

train_rows = []
test_rows = []

for file in os.listdir(DATASET_DIR):
    if file.endswith(".json"):
        file_path = os.path.join(DATASET_DIR, file)

        if "test_set" in file:
            test_rows.extend(load_qa_json(file_path))
        elif "qa_dataset" in file:
            train_rows.extend(load_qa_json(file_path))

df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)

df = df.copy()

print("Training samples:", len(df))
print("Testing samples:", len(test_df))

df.head()

Training samples: 168
Testing samples: 72


,id,question,context,answer
0,Q1,ما الحدث الذي جعل أورسون ويلز مشهورًا قبل دخول...,في ليلة 30 أكتوبر 1938، انقطع بث إذاعي ليعلن ع...,تمثيلية إذاعية عن غزو فضائي
1,Q2,ما الرواية التي استندت إليها التمثيلية الإذاعية؟,بعد الذعر الجماهيري، اكتشف الناس أن القصة مقتب...,حرب العوالم
2,Q3,لماذا اهتمت هوليوود بأورسون ويلز؟,لاحظت هوليوود أن ويلز استطاع التأثير على ملايي...,قدرته على التأثير في مشاعر الجمهور باستخدام الصوت
3,Q4,ما طبيعة أفلام هوليوود قبل Citizen Kane؟,في تلك الفترة، اعتمدت أفلام هوليوود على قوالب ...,قوالب تقليدية ونهايات سعيدة
4,Q5,لماذا تجنبت هوليوود التجريب؟,بعد الكساد العظيم، كان الجمهور يبحث عن التسلية...,رغبة الجمهور في التسلية بعد الكساد العظيم


In [23]:
df["question_norm"] = df["question"].apply(normalize_arabic)
df["context_norm"] = df["context"].apply(normalize_arabic)
df["answer_norm"] = df["answer"].apply(normalize_arabic)

df["input_text"] = "[Q] " + df["question_norm"] + " [C] " + df["context_norm"]

df[["question_norm", "context_norm", "answer_norm", "input_text"]].head()

,question_norm,context_norm,answer_norm,input_text
0,ما الحدث الذي جعل اورسون ويلز مشهورا قبل دخوله...,في ليله 30 اكتوبر 1938 انقطع بث اذاعي ليعلن عن...,تمثيليه اذاعيه عن غزو فضائي,[Q] ما الحدث الذي جعل اورسون ويلز مشهورا قبل د...
1,ما الروايه التي استندت اليها التمثيليه الاذاعيه,بعد الذعر الجماهيري اكتشف الناس ان القصه مقتبس...,حرب العوالم,[Q] ما الروايه التي استندت اليها التمثيليه الا...
2,لماذا اهتمت هوليوود باورسون ويلز,لاحظت هوليوود ان ويلز استطاع التاثير علي ملايي...,قدرته علي التاثير في مشاعر الجمهور باستخدام الصوت,[Q] لماذا اهتمت هوليوود باورسون ويلز [C] لاحظت...
3,ما طبيعه افلام هوليوود قبل Citizen Kane,في تلك الفتره اعتمدت افلام هوليوود علي قوالب ت...,قوالب تقليديه ونهايات سعيده,[Q] ما طبيعه افلام هوليوود قبل Citizen Kane [C...
4,لماذا تجنبت هوليوود التجريب,بعد الكساد العظيم كان الجمهور يبحث عن التسليه ...,رغبه الجمهور في التسليه بعد الكساد العظيم,[Q] لماذا تجنبت هوليوود التجريب [C] بعد الكساد...


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def find_best_span_tfidf(context, answer, min_len=1, max_len=8, threshold=0.25):
    context_tokens = context.split()
    answer = str(answer)

    candidate_spans = []
    positions = []

    for start in range(len(context_tokens)):
        for length in range(min_len, max_len + 1):
            end = start + length
            if end <= len(context_tokens):
                span = " ".join(context_tokens[start:end])
                candidate_spans.append(span)
                positions.append((start, end - 1))

    if not candidate_spans:
        return None, None, None, 0

    texts = [answer] + candidate_spans

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    tfidf = vectorizer.fit_transform(texts)

    sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()

    best_idx = sims.argmax()
    best_score = sims[best_idx]

    if best_score < threshold:
        return None, None, candidate_spans[best_idx], best_score

    start, end = positions[best_idx]
    return start, end, candidate_spans[best_idx], best_score

Find start/end word positions

In [25]:
def find_answer_token_positions(context, answer, debug=False):
    context_tokens = context.split()
    answer_tokens = answer.split()

    for i in range(len(context_tokens) - len(answer_tokens) + 1):
        if context_tokens[i:i+len(answer_tokens)] == answer_tokens:
            start = i
            end = i + len(answer_tokens) - 1
            return start, end

    return None, None

In [26]:
def find_answer_token_positions_smart(context, answer):
    # 1. exact match first
    start, end = find_answer_token_positions(context, answer)

    if start is not None:
        return start, end, "exact", answer, 1.0

    # 2. semantic/fuzzy fallback
    start, end, best_span, score = find_best_span_tfidf(
        context,
        answer,
        min_len=1,
        max_len=8,
        threshold=0.25
    )

    if start is not None:
        return start, end, "tfidf", best_span, score

    return None, None, "unmatched", best_span, score

Remove unmatched samples and correction for positions

In [27]:
def debug_unmatched_samples(df, max_examples=10):
    unmatched = 0

    for idx, row in df.iterrows():
        context = row["context_norm"]
        answer = row["answer_norm"]

        start, end, method, best_span, score = find_answer_token_positions_smart(
            context,
            answer
        )

        if start is None:
            unmatched += 1

            print(f"\n❌ Unmatched Sample #{unmatched}")
            print("Index:", idx)
            print("Question:", row["question_norm"])
            print("Answer:", answer)
            print("Context:", context)

            print("\n--- TOKEN VIEW ---")
            print("Context Tokens:", context.split())
            print("Answer Tokens :", answer.split())

            print("\n--- SMART MATCH RESULT ---")
            print("Best span found:", best_span)
            print("Similarity score:", score)
            print("Method:", method)

            print("=" * 80)

            if unmatched >= max_examples:
                break

    print("\nTotal unmatched samples:", unmatched)

In [28]:
print("Before removing unmatched:", len(df))

starts = []
ends = []
methods = []
matched_spans = []
scores = []
dropped_examples = []

for idx, row in df.iterrows():
    start, end, method, best_span, score = find_answer_token_positions_smart(
        row["context_norm"],
        row["answer_norm"]
    )

    starts.append(start)
    ends.append(end)
    methods.append(method)
    matched_spans.append(best_span)
    scores.append(score)

    if start is None:
        dropped_examples.append({
            "index": idx,
            "question": row["question_norm"],
            "context": row["context_norm"],
            "answer": row["answer_norm"],
            "best_span": best_span,
            "score": score
        })

# Save results
df["start_pos"] = starts
df["end_pos"] = ends
df["match_method"] = methods
df["matched_span"] = matched_spans
df["match_score"] = scores

# 📊 Stats
print("Total samples:", len(df))
print("Exact matches:", (df["match_method"] == "exact").sum())
print("Semantic matches:", (df["match_method"] == "tfidf").sum())
print("Still unmatched:", (df["start_pos"].isna()).sum())

# Drop unmatched
df = df.dropna(subset=["start_pos", "end_pos"]).copy()

df["start_pos"] = df["start_pos"].astype(int)
df["end_pos"] = df["end_pos"].astype(int)

print("After removing unmatched:", len(df))

Before removing unmatched: 168
Total samples: 168
Exact matches: 90
Semantic matches: 57
Still unmatched: 21
After removing unmatched: 147


In [29]:
def get_context_offset(question):
    return len(("[Q] " + question + " [C]").split())

df["context_offset"] = df["question_norm"].apply(get_context_offset)

df["start_label"] = df["start_pos"] + df["context_offset"]
df["end_label"] = df["end_pos"] + df["context_offset"]

df[["input_text", "answer_norm", "start_label", "end_label"]].head()

,input_text,answer_norm,start_label,end_label
0,[Q] ما الحدث الذي جعل اورسون ويلز مشهورا قبل د...,تمثيليه اذاعيه عن غزو فضائي,40,41
1,[Q] ما الروايه التي استندت اليها التمثيليه الا...,حرب العوالم,22,23
2,[Q] لماذا اهتمت هوليوود باورسون ويلز [C] لاحظت...,قدرته علي التاثير في مشاعر الجمهور باستخدام الصوت,12,17
3,[Q] ما طبيعه افلام هوليوود قبل Citizen Kane [C...,قوالب تقليديه ونهايات سعيده,16,17
4,[Q] لماذا تجنبت هوليوود التجريب [C] بعد الكساد...,رغبه الجمهور في التسليه بعد الكساد العظيم,6,13


Tokenize and pad

In [30]:
MAX_LEN = 384
VOCAB_SIZE = 10000

# Decoder answer length
answer_lengths = df["answer_norm"].apply(lambda x: len(str(x).split()))
MAX_ANSWER_LEN = int(answer_lengths.quantile(0.95)) + 2

print("MAX_ANSWER_LEN:", MAX_ANSWER_LEN)

# Create decoder texts
df["decoder_input_text"] = "[START] " + df["answer_norm"]
df["decoder_target_text"] = df["answer_norm"] + " [END]"

test_df["question_norm"] = test_df["question"].apply(normalize_arabic)
test_df["context_norm"] = test_df["context"].apply(normalize_arabic)
test_df["answer_norm"] = test_df["answer"].apply(normalize_arabic)

test_df["input_text"] = "[Q] " + test_df["question_norm"] + " [C] " + test_df["context_norm"]
test_df["decoder_input_text"] = "[START] " + test_df["answer_norm"]
test_df["decoder_target_text"] = test_df["answer_norm"] + " [END]"

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="[UNK]", filters="")

all_texts = (
    df["input_text"].tolist()
    + df["decoder_input_text"].tolist()
    + df["decoder_target_text"].tolist()
    + test_df["input_text"].tolist()
    + test_df["decoder_input_text"].tolist()
    + test_df["decoder_target_text"].tolist()
)

tokenizer.fit_on_texts(all_texts)

X = pad_sequences(
    tokenizer.texts_to_sequences(df["input_text"]),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

decoder_input = pad_sequences(
    tokenizer.texts_to_sequences(df["decoder_input_text"]),
    maxlen=MAX_ANSWER_LEN,
    padding="post",
    truncating="post"
)

decoder_target = pad_sequences(
    tokenizer.texts_to_sequences(df["decoder_target_text"]),
    maxlen=MAX_ANSWER_LEN,
    padding="post",
    truncating="post"
)

decoder_target = np.expand_dims(decoder_target, -1)

print("X:", X.shape)
print("decoder_input:", decoder_input.shape)
print("decoder_target:", decoder_target.shape)

MAX_ANSWER_LEN: 10
X: (147, 384)
decoder_input: (147, 10)
decoder_target: (147, 10, 1)


Train/validation/test split

In [31]:
# Split training data into train and validation
X_train, X_val, decoder_input_train, decoder_input_val, decoder_target_train, decoder_target_val = train_test_split(
    X,
    decoder_input,
    decoder_target,
    test_size=0.2,
    random_state=42
)

print("Train X:", X_train.shape)
print("Train decoder input:", decoder_input_train.shape)
print("Train decoder target:", decoder_target_train.shape)

print("Validation X:", X_val.shape)
print("Validation decoder input:", decoder_input_val.shape)
print("Validation decoder target:", decoder_target_val.shape)

Train X: (117, 384)
Train decoder input: (117, 10)
Train decoder target: (117, 10, 1)
Validation X: (30, 384)
Validation decoder input: (30, 10)
Validation decoder target: (30, 10, 1)


In [32]:
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df["input_text"]),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

decoder_input_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df["decoder_input_text"]),
    maxlen=MAX_ANSWER_LEN,
    padding="post",
    truncating="post"
)

decoder_target_test = pad_sequences(
    tokenizer.texts_to_sequences(test_df["decoder_target_text"]),
    maxlen=MAX_ANSWER_LEN,
    padding="post",
    truncating="post"
)

decoder_target_test = np.expand_dims(decoder_target_test, -1)

print("X_test:", X_test.shape)
print("decoder_input_test:", decoder_input_test.shape)
print("decoder_target_test:", decoder_target_test.shape)

X_test: (72, 384)
decoder_input_test: (72, 10)
decoder_target_test: (72, 10, 1)


In [33]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("decoder_input_train:", decoder_input_train.shape)
print("decoder_target_train:", decoder_target_train.shape)

print("decoder_input_val:", decoder_input_val.shape)
print("decoder_target_val:", decoder_target_val.shape)

print("decoder_input_test:", decoder_input_test.shape)
print("decoder_target_test:", decoder_target_test.shape)

Train: (117, 384)
Validation: (30, 384)
Test: (72, 384)
decoder_input_train: (117, 10)
decoder_target_train: (117, 10, 1)
decoder_input_val: (30, 10)
decoder_target_val: (30, 10, 1)
decoder_input_test: (72, 10)
decoder_target_test: (72, 10, 1)


In [34]:
# Process held-out test set using smart position matching
test_df = test_df.copy()

test_df["question_norm"] = test_df["question"].apply(normalize_arabic)
test_df["context_norm"] = test_df["context"].apply(normalize_arabic)
test_df["answer_norm"] = test_df["answer"].apply(normalize_arabic)

starts = []
ends = []

for _, row in test_df.iterrows():
    result = find_answer_token_positions_smart(
        row["context_norm"],
        row["answer_norm"]
    )

    start = result[0]
    end = result[1]

    starts.append(start)
    ends.append(end)

test_df["start_pos"] = starts
test_df["end_pos"] = ends

print("Before removing unmatched test samples:", len(test_df))
print("Matched test samples:", test_df["start_pos"].notna().sum())
print("Unmatched test samples:", test_df["start_pos"].isna().sum())

# Drop unmatched rows
test_df = test_df.dropna(subset=["start_pos", "end_pos"]).copy()

test_df["start_pos"] = test_df["start_pos"].astype(int)
test_df["end_pos"] = test_df["end_pos"].astype(int)

# Create model input AFTER filtering
test_df["input_text"] = (
    "[Q] " + test_df["question_norm"] +
    " [C] " + test_df["context_norm"]
)

# Context starts after: [Q] + question tokens + [C]
test_df["context_offset"] = test_df["question_norm"].apply(get_context_offset)

test_df["start_label"] = test_df["start_pos"] + test_df["context_offset"]
test_df["end_label"] = test_df["end_pos"] + test_df["context_offset"]

# Keep only labels inside MAX_LEN
test_df = test_df[
    (test_df["start_label"] < MAX_LEN) &
    (test_df["end_label"] < MAX_LEN)
].copy()

# Tokenize only the final filtered test_df
test_sequences = tokenizer.texts_to_sequences(test_df["input_text"])

X_test = pad_sequences(
    test_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

y_start_test = test_df["start_label"].values
y_end_test = test_df["end_label"].values

print("Final test samples:", len(test_df))
print("X_test:", X_test.shape)
print("y_start_test:", y_start_test.shape)
print("y_end_test:", y_end_test.shape)

Before removing unmatched test samples: 72
Matched test samples: 61
Unmatched test samples: 11
Final test samples: 61
X_test: (61, 384)
y_start_test: (61,)
y_end_test: (61,)


BiLSTM Extractive QA Model

In [35]:
from tensorflow.keras.layers import Concatenate
def build_bilstm_seq2seq_model(
    vocab_size,
    max_len,
    max_answer_len,
    embedding_dim=128,
    lstm_units=256
):
    # Encoder
    encoder_inputs = Input(shape=(max_len,), name="encoder_inputs")

    encoder_emb = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(encoder_inputs)

    encoder_outputs, forward_h, forward_c, backward_h, backward_c = Bidirectional(
    LSTM(lstm_units, return_sequences=True, return_state=True)
)(encoder_emb)

    state_h = Concatenate()([forward_h, backward_h])
    state_c = Concatenate()([forward_c, backward_c])

    # Decoder
    decoder_inputs = Input(shape=(max_answer_len,), name="decoder_inputs")

    decoder_emb = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(decoder_inputs)

    decoder_outputs, _, _ = LSTM(
        lstm_units * 2,
        return_sequences=True,
        return_state=True
    )(
        decoder_emb,
        initial_state=[state_h, state_c]
    )

    outputs = Dense(
        vocab_size,
        activation="softmax",
        name="answer_output"
    )(decoder_outputs)

    model = Model(
        inputs=[encoder_inputs, decoder_inputs],
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


bilstm_model = build_bilstm_seq2seq_model(
    VOCAB_SIZE,
    MAX_LEN,
    MAX_ANSWER_LEN,
    embedding_dim=128,
    lstm_units=256
)

bilstm_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 384, 128)  │  1,280,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 384)       │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ [(None, 384,      │    788,480 │ embedding_5[0][0… │
│ (Bidirectional)     │ 512), (None,      │            │ not_equal[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 10, 128)   │  1,280,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional_2[… │
│ (Concatenate)       │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional_2[… │
│ (Concatenate)       │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 10, 512), │  1,312,768 │ embedding_6[0][0… │
│                     │ (None, 512),      │            │ concatenate[0][0… │
│                     │ (None, 512)]      │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer_output       │ (None, 10, 10000) │  5,130,000 │ lstm_3[0][0]      │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,791,248 (37.35 MB)

 Trainable params: 9,791,248 (37.35 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_bilstm = bilstm_model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    validation_data=(
        [X_val, decoder_input_val],
        decoder_target_val
    ),
    epochs=15,
    batch_size=16
)

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.0072 - loss: 9.2099 - val_accuracy: 0.0967 - val_loss: 9.2084
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.0963 - loss: 9.2066 - val_accuracy: 0.0967 - val_loss: 9.2058
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.0970 - loss: 9.2024 - val_accuracy: 0.0967 - val_loss: 9.2015
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.0970 - loss: 9.1945 - val_accuracy: 0.0967 - val_loss: 9.1915
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.0968 - loss: 9.1756 - val_accuracy: 0.0967 - val_loss: 9.1599
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.0960 - loss: 9.1087 - val_accuracy: 0.0967 - val_loss: 9.0137
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.0986 - loss: 8.7640 - val_accuracy: 0.0967 - val_loss: 8.3828
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.0987 - loss: 7.5526 - val_accuracy: 0.0967 - val_loss: 7.7047
Epoch 9/

Transformer Extractive QA Model

In [ ]:
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embedding_dim):
        super().__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embedding_dim)
        self.pos_emb = Embedding(input_dim=max_len, output_dim=embedding_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [ ]:
def build_transformer_qa_model(
    vocab_size,
    max_len,
    max_answer_len,
    embedding_dim=128,
    num_heads=4,
    ff_dim=256,
    num_blocks=2
):
    encoder_inputs = Input(shape=(max_len,), name="encoder_inputs")
    decoder_inputs = Input(shape=(max_answer_len,), name="decoder_inputs")

    # =====================
    # Encoder
    # =====================
    encoder_x = TokenAndPositionEmbedding(
        max_len,
        vocab_size,
        embedding_dim
    )(encoder_inputs)

    for _ in range(num_blocks):
        attention_output = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads
        )(encoder_x, encoder_x)

        encoder_x = LayerNormalization(epsilon=1e-6)(
            encoder_x + attention_output
        )

        ffn = Dense(ff_dim, activation="relu")(encoder_x)
        ffn = Dense(embedding_dim)(ffn)

        encoder_x = LayerNormalization(epsilon=1e-6)(
            encoder_x + ffn
        )

        encoder_x = Dropout(0.2)(encoder_x)

    encoder_outputs = encoder_x

    # =====================
    # Decoder
    # =====================
    decoder_x = TokenAndPositionEmbedding(
        max_answer_len,
        vocab_size,
        embedding_dim
    )(decoder_inputs)

    for _ in range(num_blocks):
        # 1. Masked decoder self-attention
        self_attention_output = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads
        )(
            decoder_x,
            decoder_x,
            use_causal_mask=True
        )

        decoder_x = LayerNormalization(epsilon=1e-6)(
            decoder_x + self_attention_output
        )

        # 2. Cross-attention: decoder attends to encoder output
        cross_attention_output = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads
        )(
            decoder_x,
            encoder_outputs
        )

        decoder_x = LayerNormalization(epsilon=1e-6)(
            decoder_x + cross_attention_output
        )

        # 3. Feed-forward
        ffn = Dense(ff_dim, activation="relu")(decoder_x)
        ffn = Dense(embedding_dim)(ffn)

        decoder_x = LayerNormalization(epsilon=1e-6)(
            decoder_x + ffn
        )

        decoder_x = Dropout(0.2)(decoder_x)

    outputs = Dense(
        vocab_size,
        activation="softmax",
        name="answer_output"
    )(decoder_x)

    model = Model(
        inputs=[encoder_inputs, decoder_inputs],
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


transformer_model = build_transformer_qa_model(
    VOCAB_SIZE,
    MAX_LEN,
    MAX_ANSWER_LEN,
    embedding_dim=128,
    num_heads=4,
    ff_dim=256,
    num_blocks=2
)

transformer_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 384, 128)  │  1,329,152 │ encoder_inputs[0… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 384, 128)  │     66,048 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 384, 128)  │          0 │ token_and_positi… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 384, 128)  │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 384, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 384, 128)  │     32,896 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 384, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 384, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 384, 128)  │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 384, 128)  │     66,048 │ dropout_3[0][0],  │
│ (MultiHeadAttentio… │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 384, 128)  │          0 │ dropout_3[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 384, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 384, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 10, 128)   │  1,281,280 │ decoder_inputs[0… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 384, 128)  │     32,896 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │     66,048 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,562,960 (17.41 MB)

 Trainable params: 4,562,960 (17.41 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_transformer = transformer_model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    validation_data=(
        [X_val, decoder_input_val],
        decoder_target_val
    ),
    epochs=30,
    batch_size=16
)

Plot curves

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(title + " Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["start_output_accuracy"], label="Start Train Acc")
    plt.plot(history.history["val_start_output_accuracy"], label="Start Val Acc")
    plt.plot(history.history["end_output_accuracy"], label="End Train Acc")
    plt.plot(history.history["val_end_output_accuracy"], label="End Val Acc")
    plt.title(title + " Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_history(history_bilstm, "BiLSTM QA")
plot_history(history_transformer, "Transformer QA")

Evaluate both models

In [ ]:
print("BiLSTM Evaluation:")
bilstm_model.evaluate(
    X_test,
    {
        "start_output": y_start_test,
        "end_output": y_end_test
    }
)

print("Transformer Evaluation:")

transformer_model.evaluate(
    [X_test, decoder_input_test],
    decoder_target_test
)

Inference function

In [ ]:
def generate_answer_transformer(question, context, model, tokenizer, max_len=MAX_LEN, max_answer_len=MAX_ANSWER_LEN):
    question_norm = normalize_arabic(question)
    context_norm = normalize_arabic(context)

    input_text = "[Q] " + question_norm + " [C] " + context_norm

    encoder_seq = tokenizer.texts_to_sequences([input_text])
    encoder_seq = pad_sequences(
        encoder_seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    start_token = tokenizer.word_index.get("[START]")
    end_token = tokenizer.word_index.get("[END]")

    index_word = {v: k for k, v in tokenizer.word_index.items()}

    decoded_tokens = [start_token]

    for i in range(max_answer_len - 1):
        decoder_seq = pad_sequences(
            [decoded_tokens],
            maxlen=max_answer_len,
            padding="post",
            truncating="post"
        )

        predictions = model.predict(
            [encoder_seq, decoder_seq],
            verbose=0
        )

        next_token_id = np.argmax(predictions[0, i, :])

        if next_token_id == end_token or next_token_id == 0:
            break

        decoded_tokens.append(next_token_id)

    words = []

    for token_id in decoded_tokens[1:]:
        word = index_word.get(token_id, "")

        if word in ["[START]", "[END]", "[UNK]", ""]:
            continue

        words.append(word)

    return " ".join(words)

In [ ]:
def generate_answer_bilstm(question, context, model, tokenizer, max_len=MAX_LEN, max_answer_len=MAX_ANSWER_LEN):
    question_norm = normalize_arabic(question)
    context_norm = normalize_arabic(context)

    input_text = "[Q] " + question_norm + " [C] " + context_norm

    encoder_seq = tokenizer.texts_to_sequences([input_text])
    encoder_seq = pad_sequences(
        encoder_seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    start_token = tokenizer.word_index.get("[START]")
    end_token = tokenizer.word_index.get("[END]")

    index_word = {v: k for k, v in tokenizer.word_index.items()}

    decoded_tokens = [start_token]

    for i in range(max_answer_len - 1):
        decoder_seq = pad_sequences(
            [decoded_tokens],
            maxlen=max_answer_len,
            padding="post",
            truncating="post"
        )

        predictions = model.predict(
            [encoder_seq, decoder_seq],
            verbose=0
        )

        next_token_id = np.argmax(predictions[0, i, :])

        if next_token_id == end_token or next_token_id == 0:
            break

        decoded_tokens.append(next_token_id)

    words = []

    for token_id in decoded_tokens[1:]:
        word = index_word.get(token_id, "")

        if word in ["[START]", "[END]", "[UNK]", ""]:
            continue

        words.append(word)

    return " ".join(words)

In [ ]:
sample = test_df.iloc[0]

predicted_answer = generate_answer_bilstm(
    sample["question"],
    sample["context"],
    bilstm_model,
    tokenizer
)

print("Question:", sample["question"])
print("True Answer:", sample["answer"])
print("Predicted Answer:", predicted_answer)

In [ ]:

sample = test_df.iloc[0]

predicted_answer = generate_answer_transformer(
    sample["question"],
    sample["context"],
    transformer_model,
    tokenizer
)

print("Question:", sample["question"])
print("True Answer:", sample["answer"])
print("Predicted Answer:", predicted_answer)